### Cómo usar esta práctica

Esta práctica recorre **los mismos ocho puntos de decisión que evalúa el examen**, en el mismo orden.
Lo que cambiará en el examen es el conjunto de datos: **las propiedades serán distintas y, por lo tanto,
varias decisiones correctas serán otras**.


Los dos **retos finales** son deliberadamente distintos al resto: te muestran casos en los que la respuesta
del día **no aplica**. Ese formato sí aparecerá en el examen.

---

### Contexto clínico

Registro de una **cohorte de insuficiencia cardiaca crónica**. Cada fila es un paciente.
Objetivo: predecir `reingreso_30d`.

| Variable | Tipo | Descripción |
|---|---|---|
| `id_paciente` | Texto | Identificador |
| `edad` | Numérica | Años cumplidos |
| `sexo` | Categórica nominal | M / F |
| `frec_cardiaca` | Numérica | Frecuencia cardiaca (lpm) |
| `presion_sistolica` | Numérica | Presión arterial sistólica (mmHg) |
| `bnp` | Numérica | Péptido natriurético tipo B (pg/mL) |
| `creatinina` | Numérica | Creatinina sérica (mg/dL) |
| `colesterol_total` | Numérica | Colesterol total (mg/dL) |
| `fevi_ingreso` | Numérica | Fracción de eyección (%) — *reservada para el reto final* |
| `tipo_dolor` | Categórica nominal | TA / AA / DNA / ASINT |
| `pendiente_st` | Categórica nominal | Ascendente / Plana / Descendente |
| `reingreso_30d` | Binaria | **Variable objetivo** |


In [1]:
# ============================================================
#  CELDA DADA — ejecútala sin modificarla
# ============================================================
import warnings, os
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score, recall_score)

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": .3, "font.size": 9})

SEMILLA = 42


def lambda_ic(x, alpha=0.05):
    """Lambda de Yeo-Johnson por maxima verosimilitud + IC 95 % (verosimilitud perfilada).

    Devuelve (lambda, limite_inferior, limite_superior).
    Si el IC contiene 1.0, la transformacion NO es estadisticamente necesaria.
    """
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    lam = stats.yeojohnson_normmax(x)
    corte = stats.yeojohnson_llf(lam, x) - 0.5 * stats.chi2.ppf(1 - alpha, 1)
    malla = np.linspace(lam - 6, lam + 6, 2001)
    llf = np.array([stats.yeojohnson_llf(l, x) for l in malla])
    dentro = malla[llf >= corte]
    return lam, dentro.min(), dentro.max()


def perfil(df, columnas):
    """Tabla diagnostica: sesgo, minimo, negativos, ceros, lambda e IC 95 %."""
    filas = []
    for c in columnas:
        s = df[c].dropna().astype(float)
        lam, lo, hi = lambda_ic(s)
        filas.append({
            "variable": c,
            "sesgo": round(float(stats.skew(s)), 2),
            "min": round(float(s.min()), 2),
            "negativos": int((s < 0).sum()),
            "ceros": int((s == 0).sum()),
            "lambda": round(lam, 3),
            "IC_inf": round(lo, 2),
            "IC_sup": round(hi, 2),
            "IC_contiene_1": bool(lo <= 1 <= hi),
        })
    return pd.DataFrame(filas)


def cargar(nombre):
    """Busca el CSV junto al notebook o en ./datos/."""
    for r in (nombre, os.path.join("datos", nombre),
              os.path.join("..", "datos", nombre)):
        if os.path.exists(r):
            return pd.read_csv(r)
    raise FileNotFoundError(
        f"No encuentro '{nombre}'. Colocalo en la misma carpeta que este notebook.")


print("Entorno listo.")

Entorno listo.


---
## Punto 0 — Ingestión e inspección inicial
*Subtema 3.2*

In [2]:
df = cargar("cohorte_ic_repaso.csv")

print("Dimensiones:", df.shape)
print()
print(df.dtypes.to_string())
print()
print("Distribución del desenlace:")
print(df["reingreso_30d"].value_counts(normalize=True).round(3).to_string())
df.head()

Dimensiones: (900, 12)

id_paciente           object
edad                 float64
sexo                  object
frec_cardiaca        float64
presion_sistolica    float64
bnp                  float64
creatinina           float64
colesterol_total     float64
fevi_ingreso         float64
tipo_dolor            object
pendiente_st          object
reingreso_30d          int64

Distribución del desenlace:
reingreso_30d
1    0.508
0    0.492


,id_paciente,edad,sexo,frec_cardiaca,presion_sistolica,bnp,creatinina,colesterol_total,fevi_ingreso,tipo_dolor,pendiente_st,reingreso_30d
0,IC-00001,37.0,M,75.0,124.0,545.8,0.60,NaN,62.0,ASINT,Descendente,0
1,IC-00002,64.0,F,80.0,116.0,370.0,0.93,146.0,43.0,AA,Ascendente,0
2,IC-00003,76.0,M,83.0,139.0,91.1,1.33,NaN,57.0,AA,Ascendente,0
3,IC-00004,82.0,M,88.0,109.0,474.2,1.35,263.0,38.0,DNA,Ascendente,1
4,IC-00005,72.0,F,78.0,155.0,154.3,0.63,134.0,50.0,TA,Ascendente,0


**Anota tres observaciones** de esta primera inspección que puedan condicionar decisiones posteriores.

> 1.Nueve variables numéricas y tres de texto: las tres de texto tendrán que codificarse antes de modelar.
2. El desenlace está **razonablemente balanceado** (≈ 50 % de reingresos). Esto tendrá consecuencias en el Punto 8: aquí la accuracy sí es una métrica informativa.
3. `colesterol_total` es la única con valores faltantes visibles, y su mínimo es sospechosamente bajo.


---
## Punto 1 — Integridad y segregación de datos
*Subtemas 3.2 y 3.4*

In [3]:
# Antes de partir, siempre se verifica la unidad de observación
print("Filas:            ", len(df))
print("Pacientes únicos: ", df["id_paciente"].nunique())
print("Duplicados:       ", len(df) - df["id_paciente"].nunique())

Filas:             900
Pacientes únicos:  900
Duplicados:        0


**Tarea.** Construye `df_tr` y `df_te` con `test_size=0.25`, `random_state=SEMILLA` y
estratificando por el desenlace.

> Pregúntale al dato: *¿la fila y la unidad de observación son la misma cosa?* Aquí sí. **No siempre lo son.**

In [8]:
# ============================= TU TURNO =============================
# Pista: aquí cada fila es un paciente distinto, así que la partición
#        estándar por filas es válida.
df_tr, df_te = train_test_split(
    df,
    test_size=0.25,      # 25% para prueba
    stratify=df["reingreso_30d"],          # conserva la proporción de clases
    random_state=SEMILLA    # semilla fija -> división reproducible
)

In [9]:
print("Filas train / test:", len(df_tr), "/", len(df_te))
print("Prevalencia train: %.3f" % df_tr["reingreso_30d"].mean())
print("Prevalencia test:  %.3f" % df_te["reingreso_30d"].mean())

Filas train / test: 675 / 225
Prevalencia train: 0.508
Prevalencia test:  0.507



> **Tu justificación:**
>
> Prevalencia train y test salen muy parecidos en sus valores, cada uno siendo: 0.508 y 0.507 respectivamente


---
## Punto 2 — Selección de funciones: variables admisibles
*Subtemas 3.1 y 4.1*

In [10]:
numericas = df_tr.select_dtypes(include=np.number).columns.drop("reingreso_30d")
print("Correlación de cada variable numérica con el desenlace:")
print(df_tr[numericas].corrwith(df_tr["reingreso_30d"]).sort_values(key=abs, ascending=False).round(3).to_string())

Correlación de cada variable numérica con el desenlace:
bnp                  0.291
creatinina           0.262
edad                 0.170
presion_sistolica   -0.124
frec_cardiaca        0.104
fevi_ingreso        -0.063
colesterol_total     0.040


**Tarea.** Revisa la tabla y responde: ¿hay alguna variable que **no debas** usar?

> Pregúntale al dato: *¿alguna correlación es sospechosamente alta?* Y sobre todo: *¿alguna variable se
> registra en un momento posterior al que el modelo pretende predecir?*


> **Tu justificación:**
>
>no hay ninguna correlacion alta, 


In [ ]:
# fevi_ingreso se reserva para el reto final: se retira del flujo principal
df_tr = df_tr.drop(columns=["fevi_ingreso"])
df_te = df_te.drop(columns=["fevi_ingreso"])
print("Columnas en el flujo principal:", df_tr.shape[1])

---
## Punto 3 — Valores atípicos y errores de captura
*Subtema 4.3*


In [ ]:
def limites_iqr(s):
    q1, q3 = s.quantile([.25, .75]); r = q3 - q1
    return q1 - 1.5 * r, q3 + 1.5 * r

for v in ["colesterol_total", "bnp"]:
    s = df_tr[v].dropna()
    lo, hi = limites_iqr(s)
    print(f"{v:20s} límites IQR = [{lo:8.2f}, {hi:8.2f}]   atípicos = {int(((s<lo)|(s>hi)).sum()):4d}   mínimo = {s.min():.2f}")

print("\ncolesterol_total -> registros en cero:", int((df_tr["colesterol_total"] == 0).sum()))
print("\nEfecto de los ceros sobre la mediana de colesterol_total:")
print("  mediana CON los ceros:", df_tr["colesterol_total"].median())
print("  mediana SIN los ceros:", df_tr.loc[df_tr["colesterol_total"] != 0, "colesterol_total"].median())

print("\nbnp: los 8 valores más altos, junto a la creatinina del mismo paciente")
print(df_tr.nlargest(8, "bnp")[["bnp", "creatinina", "reingreso_30d"]].to_string(index=False))

**Tarea.** El criterio IQR marca valores en las dos variables. Decide qué hacer con cada una.

> Pregúntale al dato: *¿este valor es raro, o es imposible?* Rareza estadística ≠ invalidez clínica.

In [ ]:
# ============================= TU TURNO =============================
# colesterol_total == 0  ->  ¿error o valor real?
# bnp muy alto           ->  ¿error o valor real?
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación:**
>
> _Escribe aquí, citando la evidencia numérica._


---
## Punto 4 — Valores faltantes
*Subtema 3.3*

In [ ]:
print("Porcentaje de faltantes (entrenamiento):")
print((df_tr.isna().mean() * 100).round(1).loc[lambda s: s > 0].to_string())

print("\n¿El patrón de ausencia depende de otras variables?")
falta = df_tr["colesterol_total"].isna()
print(df_tr.groupby(falta)[["edad", "bnp", "creatinina", "reingreso_30d"]].mean().round(2).to_string())
print("\n(fila False = colesterol presente | fila True = colesterol ausente)")

**Tarea.** Imputa los faltantes de `colesterol_total`.

> Pregúntale al dato: *¿el hecho de que falte me dice algo?* Compara las dos filas de la tabla anterior.

In [ ]:
# ============================= TU TURNO =============================
# Pista: si los perfiles de ambos grupos son parecidos, la ausencia
#        no aporta información y basta una imputación simple.
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación:**
>
> _Escribe aquí, citando la evidencia numérica._


---
## Punto 5 — Codificación de variables categóricas
*Subtemas 4.4 y 4.7*

In [ ]:
for v in ["sexo", "tipo_dolor", "pendiente_st"]:
    print(f"{v:16s} niveles = {df_tr[v].nunique()}  ->  {sorted(df_tr[v].unique())}")

print("\nTasa de reingreso por nivel de pendiente_st:")
print(df_tr.groupby("pendiente_st")["reingreso_30d"].agg(["size", "mean"]).round(3).to_string())

**Tarea.** Codifica las tres variables categóricas.

> Pregúntale al dato: *¿los niveles tienen un orden real?* Y: *¿cuántos niveles hay?*
>
> Mira con cuidado `pendiente_st`. El nombre sugiere una progresión (Ascendente → Plana → Descendente),
> pero la tabla de arriba muestra la tasa de reingreso por nivel. ¿Es monótona?

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación:**
>
> _Escribe aquí, citando la evidencia numérica._


---
## Punto 6 — Corrección del sesgo
*Subtema 4.5 — el punto de mayor peso del examen*

<div style="border-left:5px solid #0FA3B1;background:#f4f7f9;padding:12px 16px;border-radius:5px;font-family:Calibri,Arial,sans-serif;">
<b style="color:#1E2664;">Las tres preguntas antes de transformar</b><br><b>1. &iquest;Cu&aacute;l es la <u>direcci&oacute;n</u> del sesgo?</b> Positivo = derecho; negativo = izquierdo.<br><b>2. &iquest;El <u>dominio</u> admite el logaritmo?</b> Exige valores estrictamente positivos: sin ceros ni negativos.<br><b>3. &iquest;La correcci&oacute;n es <u>necesaria</u>?</b> Si el IC 95 % de &lambda; contiene 1.0, no puede rechazarse que &lambda; = 1, y &lambda; = 1 es la identidad: no transformar.
</div>

In [ ]:
objetivo = ["bnp", "creatinina", "frec_cardiaca", "edad"]
print(perfil(df_tr, objetivo).to_string(index=False))

fig, ax = plt.subplots(1, 4, figsize=(13, 2.6))
for a, v in zip(ax, objetivo):
    a.hist(df_tr[v], bins=35, color="#0FA3B1")
    a.set_title(f"{v}\nsesgo = {stats.skew(df_tr[v]):+.2f}", fontsize=8)
plt.tight_layout(); plt.show()

**Tarea.** Decide qué hacer con cada una de las cuatro variables. **No todas requieren lo mismo, y
al menos una no requiere nada.**

In [ ]:
# ============================= TU TURNO =============================
# bnp              -> ?
# creatinina       -> ?
# frec_cardiaca    -> ?
# edad             -> ?
# ---- ESCRIBE TU CÓDIGO AQUÍ ----


> **Tu justificación.** Completa la tabla:
>
> | Variable | Decisión | Evidencia numérica |
> |---|---|---|
> | `bnp` | | |
> | `creatinina` | | |
> | `frec_cardiaca` | | |
> | `edad` | | |


---
## Punto 7 — Escalamiento
*Subtema 4.6*

In [ ]:
num_final = ["edad", "frec_cardiaca", "presion_sistolica", "bnp", "creatinina", "colesterol_total"]
print(df_tr[num_final].describe().T[["mean", "std", "min", "50%", "max"]].round(2).to_string())
print("\nSesgo residual:")
print(df_tr[num_final].apply(lambda s: round(float(stats.skew(s)), 2)).to_string())

**Tarea.** Escala las variables numéricas.

> Pregúntale al dato: *¿queda alguna cola pesada que arrastraría la media y la desviación estándar?*

In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----



> **Tu justificación:**
>
> _Escribe aquí, citando la evidencia numérica._


---
## Punto 8 — Entrenamiento y elección de la métrica
*Subtemas 3.5 y 3.1*

In [ ]:
descartar = ["id_paciente", "reingreso_30d"]
X_tr = df_tr.drop(columns=[c for c in descartar if c in df_tr.columns])
X_te = df_te.drop(columns=[c for c in descartar if c in df_te.columns])
y_tr, y_te = df_tr["reingreso_30d"], df_te["reingreso_30d"]

assert X_tr.select_dtypes(exclude=np.number).empty, "Quedan columnas no numéricas"
print("Matriz final -> train:", X_tr.shape, "| test:", X_te.shape)
print("Prevalencia en prueba: %.4f" % y_te.mean())

modelo = LogisticRegression(solver="lbfgs", max_iter=3000, random_state=SEMILLA).fit(X_tr, y_tr)
pred = modelo.predict(X_te); prob = modelo.predict_proba(X_te)[:, 1]

print("\n--- MODELO ------------------------------------------------------")
print("Accuracy : %.4f" % accuracy_score(y_te, pred))
print("Recall   : %.4f" % recall_score(y_te, pred))
print("AUC ROC  : %.4f" % roc_auc_score(y_te, prob))
print("\nMatriz de confusión [fila = real, columna = predicho]:")
print(confusion_matrix(y_te, pred))
print("\n", classification_report(y_te, pred, digits=3))
print("--- Referencia: clasificador trivial 'nadie reingresa' ----------")
print("Accuracy : %.4f" % accuracy_score(y_te, np.zeros_like(y_te)))

**Tarea.** Responde en prosa:

1. Compara la accuracy del modelo con la del clasificador trivial. ¿Cuánto aporta el modelo?
2. ¿Qué métrica reportarías aquí, y por qué?


> **Tu justificación:**
>
> _Escribe aquí, citando la evidencia numérica._


---
# Retos finales

<div style="border-left:5px solid #E8912B;background:#f4f7f9;padding:12px 16px;border-radius:5px;font-family:Calibri,Arial,sans-serif;">
<b style="color:#1E2664;">Cambio de formato</b><br>Los dos retos siguientes presentan variables donde <b>la respuesta de hoy no aplica</b>. Este es el tipo de reactivo que aparecer&aacute; en el examen. Resu&eacute;lvelos aunque no alcance el tiempo de clase.
</div>

## Reto A — Una variable con sesgo izquierdo

`fevi_ingreso` (fracción de eyección) es la variable que reservamos al inicio.

In [ ]:
crudo = cargar("cohorte_ic_repaso.csv")
fevi = crudo["fevi_ingreso"]

print("sesgo:", round(float(stats.skew(fevi)), 2))
print("rango: [%.0f, %.0f]" % (fevi.min(), fevi.max()))
lam, lo, hi = lambda_ic(fevi)
print("lambda = %+.2f   IC 95%% = [%+.2f, %+.2f]" % (lam, lo, hi))

fig, ax = plt.subplots(1, 3, figsize=(12, 2.6))
ax[0].hist(fevi, bins=35, color="#1E2664"); ax[0].set_title(f"original (sesgo {stats.skew(fevi):+.2f})", fontsize=8)
ax[1].hist(np.log(fevi), bins=35, color="#E8912B"); ax[1].set_title(f"log (sesgo {stats.skew(np.log(fevi)):+.2f})", fontsize=8)
yj = stats.yeojohnson(fevi.to_numpy(float))[0]
ax[2].hist(yj, bins=35, color="#0FA3B1"); ax[2].set_title(f"Yeo-Johnson (sesgo {stats.skew(yj):+.2f})", fontsize=8)
plt.tight_layout(); plt.show()

**Preguntas.**

1. ¿El logaritmo corrige el sesgo de esta variable? Mira el histograma central y su cifra de sesgo.
2. ¿Por qué λ resulta **mayor** que 1 aquí, cuando en `bnp` resultó ≈ 0?
3. Formula la regla general en una frase.

> **Tu respuesta:**


## Reto B — Una variable con valores negativos

In [ ]:
# Variable derivada: desviación del BNP respecto al umbral clínico de referencia (400 pg/mL)
delta_bnp = crudo["bnp"] - 400

print("sesgo:", round(float(stats.skew(delta_bnp)), 2))
print("mínimo: %.1f  |  negativos: %d  |  ceros: %d"
      % (delta_bnp.min(), int((delta_bnp < 0).sum()), int((delta_bnp == 0).sum())))

print("\n¿Qué ocurre si aplicamos np.log() ?")
with np.errstate(invalid="ignore", divide="ignore"):
    intento = np.log(delta_bnp)
print("  valores no finitos generados:", int((~np.isfinite(intento)).sum()), "de", len(delta_bnp))

**Tarea.** El sesgo es positivo y fuerte, así que "hay que transformar". Pero `np.log()` destruye
la mayor parte de la columna.

Aplica la transformación adecuada y comprueba el sesgo resultante.


In [ ]:
# ============================= TU TURNO =============================
# ---- ESCRIBE TU CÓDIGO AQUÍ ----


> **Tu conclusión en una frase:**


---
# Mapa de decisiones — guía de estudio

**Esta tabla es lo que debes estudiar.** No las celdas de código: **las preguntas de la columna central**.

| # | Punto de decisión | La pregunta que le haces al dato | Hoy la respuesta fue |
|---|---|---|---|
| 1 | Integridad y segregación | ¿La fila y la unidad de observación son la misma cosa? | Sí (900 filas = 900 pacientes) → partición estándar |
| 2 | Variables admisibles | ¿Alguna variable se registra **después** del desenlace? | No → no se elimina nada |
| 3 | Atípicos | ¿Este valor es **raro** o es **imposible**? | Colesterol 0 = imposible (corregir) · BNP alto = real (conservar) |
| 4 | Faltantes | ¿El **hecho de que falte** me dice algo? | No (MCAR) → imputación simple, sin indicador |
| 5 | Codificación | ¿Los niveles tienen un **orden monótono** verificable? | No → one-hot para las tres |
| 6 | Corrección del sesgo | ¿Qué **dirección**? ¿El **dominio** admite log? ¿El **IC de λ** excluye a 1? | Log en `bnp` y `creatinina` · sin transformar en las otras dos |
| 7 | Escalamiento | ¿Queda alguna **cola pesada** que arrastre la media? | No → `StandardScaler` |
| 8 | Métrica | ¿Cuánto supero al **clasificador trivial**? | Bastante (≈ 50 % de prevalencia) → accuracy interpretable |

### En el examen

Los ocho puntos serán **los mismos** y en **el mismo orden**. El conjunto de datos será distinto, y por eso
**varias respuestas de la última columna serán otras**. El examen es enteramente práctico y dura 90 minutos.

Se te calificará en tres componentes por punto: **ejecución** (el código corre), **decisión** (elegiste lo que
el dato exige) y **justificación** (citaste la evidencia numérica). Una decisión correcta sin evidencia que la
respalde vale la mitad, y una decisión equivocada que tú mismo detectes y corrijas conserva puntaje parcial.

### Lista de verificación antes del examen

- [ ] Sé comprobar si hay registros duplicados por sujeto, y qué hacer si los hay.
- [ ] Sé distinguir un error de captura de un valor extremo clínicamente válido.
- [ ] Sé comparar el perfil de los casos con y sin dato faltante para decidir si la ausencia es informativa.
- [ ] Sé verificar si una variable categórica tiene orden monótono antes de codificarla.
- [ ] Sé leer la **dirección** del sesgo, no solo su magnitud.
- [ ] Sé revisar el mínimo y el conteo de negativos antes de intentar un logaritmo.
- [ ] Sé interpretar el IC 95 % de λ para decidir **no** transformar.
- [ ] Sé calcular el desempeño del clasificador trivial y comparar contra él.
